# Region-level OCR (third review C5/M5), Colab CPU

Cuts every PAGE-XML text region out of the 40 evaluation pages and reads it with Tesseract and EasyOCR
(`ocr/run_region_ocr.py`). It first re-OCRs the whole pages and checks that they match the stored OCR of
the main run (`ocr_eval40.jsonl`), so the engines are the same versions; if they do not match it stops.

**Runtime:** CPU is enough (Runtime > Change runtime type > CPU). About 1.5-2.5 h; it resumes if rerun.

**Needs on Google Drive:** `My Drive/Dataset/REID2019.zip` (as before) and
`My Drive/bengali-postocr-results/ocr_eval40.jsonl`. Output goes to `My Drive/bengali-postocr-results/`.


In [ ]:
import os, subprocess
REPO_DIR = '/content/bengali-postocr-sllm'
if not os.path.isdir(REPO_DIR):
    !git clone -q https://github.com/Ariffurrhmn/bengali-postocr-sllm.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull -q
!cd {REPO_DIR} && git log --oneline -1
!apt-get -qq update && apt-get -qq install -y tesseract-ocr > /dev/null
!pip install -q pytesseract easyocr==1.7.2 jiwer
!mkdir -p {REPO_DIR}/.tessdata
!curl -sL -o {REPO_DIR}/.tessdata/ben.traineddata https://github.com/tesseract-ocr/tessdata_best/raw/main/ben.traineddata
!tesseract --version | head -1


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RESULTS_DIR = '/content/drive/My Drive/bengali-postocr-results'
DATASET_DIR = '/content/Competition_dataset_ImagesPAGEXML'
!unzip -q -o "/content/drive/My Drive/Dataset/REID2019.zip" -d /content
assert os.path.exists(f'{RESULTS_DIR}/ocr_eval40.jsonl'), 'copy ocr_eval40.jsonl to My Drive/bengali-postocr-results/'


In [ ]:
os.environ['TESSDATA_PREFIX'] = f'{REPO_DIR}/.tessdata'
os.environ['TESSERACT_CMD'] = 'tesseract'
# --verify re-OCRs the whole pages first and stops if Tesseract differs from the main run.
!cd {REPO_DIR}/ocr && python run_region_ocr.py --dataset-dir "{DATASET_DIR}" \
    --stored "{RESULTS_DIR}/ocr_eval40.jsonl" --verify --out "{RESULTS_DIR}/ocr_region40.jsonl"


In [ ]:
import json
v = json.load(open(f'{RESULTS_DIR}/ocr_region40_verify.json'))
print('versions', v['versions'], '| identical pages: Tesseract', v['tesseract_identical'], 'EasyOCR', v['easyocr_identical'])
rows = [json.loads(l) for l in open(f'{RESULTS_DIR}/ocr_region40.jsonl', encoding='utf-8') if l.strip()]
print(len(rows), 'pages,', sum(len(r['regions']) for r in rows), 'regions')
